In [ ]:
# ============================================================
# TALTECH SEMANTIC PIPELINE
# NOTEBOOK 02B — RECONCILIATION EVIDENCE SEARCH
# ============================================================

from pathlib import Path
import pandas as pd
import re

# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"

EXTRACTION_DIR = OUTPUT_DIR / "01_extraction"
RECON_DIR = OUTPUT_DIR / "02_reconciliation"

RECON_DIR.mkdir(
    parents=True,
    exist_ok=True
)

IFC_FILE = DATA_DIR / "DS3_TalTech_V4.ifc"

DEVICES_FILE = (
    EXTRACTION_DIR /
    "01_selected_devices.csv"
)

print("Project directory :", PROJECT_DIR)
print("IFC model         :", IFC_FILE)
print("Selected devices  :", DEVICES_FILE)
print("Output directory  :", RECON_DIR)

assert IFC_FILE.exists(), (
    f"IFC file not found: {IFC_FILE}"
)

assert DEVICES_FILE.exists(), (
    f"Selected devices file not found: {DEVICES_FILE}"
)

print("\n✓ Input files found")

# ============================================================
# 2. LOAD THE 7 OPERATIONAL UUIDs
# ============================================================

devices_df = pd.read_csv(
    DEVICES_FILE,
    dtype=str
)

devices = (
    devices_df["device_id"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

devices = sorted(devices)

print("\n====================================")
print("OPERATIONAL DEVICES")
print("====================================")

print("Number of UUIDs:", len(devices))

for uuid in devices:
    print(" -", uuid)

# ============================================================
# 3. LOAD IFC WITH IFCOPENSHELL
# ============================================================

try:
    import ifcopenshell
    import ifcopenshell.util.element

except ImportError:

    raise ImportError(
        "\nIfcOpenShell is required for this notebook.\n"
        "Install it with:\n\n"
        "pip install ifcopenshell\n"
    )

ifc = ifcopenshell.open(
    str(IFC_FILE)
)

print("\n✓ IFC successfully loaded")

# ============================================================
# 4. BASIC IFC INVENTORY
# ============================================================

entity_types = [
    "IfcProject",
    "IfcSite",
    "IfcBuilding",
    "IfcBuildingStorey",
    "IfcSpace",
    "IfcSensor",
    "IfcFlowMeter",
    "IfcDistributionSystem",
    "IfcAirTerminal",
    "IfcUnitaryEquipment",
    "IfcAirToAirHeatRecovery",
    "IfcBuildingElementProxy"
]

inventory = []

for entity_type in entity_types:

    try:
        count = len(
            ifc.by_type(entity_type)
        )

    except Exception:
        count = None

    inventory.append(
        {
            "entity_type": entity_type,
            "count": count
        }
    )

inventory_df = pd.DataFrame(
    inventory
)

inventory_df.to_csv(
    RECON_DIR /
    "05_ifc_entity_inventory_extended.csv",
    index=False
)

print("\n====================================")
print("IFC INVENTORY")
print("====================================")

print(
    inventory_df.to_string(
        index=False
    )
)

# ============================================================
# 5. EXTRACT IFC SPACES + PROPERTIES
# ============================================================

space_rows = []

spaces = ifc.by_type(
    "IfcSpace"
)

for space in spaces:

    try:
        psets = (
            ifcopenshell
            .util
            .element
            .get_psets(space)
        )

    except Exception:
        psets = {}

    # Flatten properties into searchable text
    property_text_parts = []

    for pset_name, properties in psets.items():

        property_text_parts.append(
            str(pset_name)
        )

        if isinstance(
            properties,
            dict
        ):

            for key, value in properties.items():

                property_text_parts.append(
                    str(key)
                )

                property_text_parts.append(
                    str(value)
                )

    property_text = " | ".join(
        property_text_parts
    )

    space_rows.append(
        {
            "ifc_global_id":
                getattr(
                    space,
                    "GlobalId",
                    ""
                ),

            "ifc_name":
                getattr(
                    space,
                    "Name",
                    ""
                ),

            "ifc_long_name":
                getattr(
                    space,
                    "LongName",
                    ""
                ),

            "ifc_object_type":
                getattr(
                    space,
                    "ObjectType",
                    ""
                ),

            "properties":
                property_text
        }
    )

spaces_df = pd.DataFrame(
    space_rows
)

spaces_df.to_csv(
    RECON_DIR /
    "06_ifc_space_evidence.csv",
    index=False
)

print("\n====================================")
print("IFC SPACES")
print("====================================")

print(
    "Number of IfcSpace entities:",
    len(spaces_df)
)

print(
    spaces_df[
        [
            "ifc_global_id",
            "ifc_name",
            "ifc_long_name"
        ]
    ]
    .head(20)
    .to_string(
        index=False
    )
)

# ============================================================
# 6. SEARCH TARGET ROOM REFERENCES
# ============================================================

# These room references appeared during the previous
# TalTech IFC exploration.
#
# IMPORTANT:
# Finding the rooms does NOT establish UUID → room mapping.

TARGET_ROOM_REFERENCES = [
    "206",
    "207",
    "309"
]

target_room_rows = []

for _, row in spaces_df.iterrows():

    searchable = " ".join(
        [
            str(row["ifc_name"]),
            str(row["ifc_long_name"]),
            str(row["ifc_object_type"]),
            str(row["properties"])
        ]
    ).lower()

    matches = []

    for room_ref in TARGET_ROOM_REFERENCES:

        pattern = (
            r"(?<!\d)"
            + re.escape(room_ref)
            + r"(?!\d)"
        )

        if re.search(
            pattern,
            searchable
        ):
            matches.append(
                room_ref
            )

    if matches:

        result = row.to_dict()

        result[
            "matched_room_reference"
        ] = ",".join(matches)

        target_room_rows.append(
            result
        )

target_rooms_df = pd.DataFrame(
    target_room_rows
)

target_rooms_df.to_csv(
    RECON_DIR /
    "07_target_room_evidence.csv",
    index=False
)

print("\n====================================")
print("TARGET ROOM REFERENCES")
print("====================================")

if len(target_rooms_df) == 0:

    print(
        "No target room references found."
    )

else:

    print(
        target_rooms_df[
            [
                "ifc_global_id",
                "ifc_name",
                "ifc_long_name",
                "matched_room_reference"
            ]
        ]
        .to_string(
            index=False
        )
    )

# ============================================================
# 7. EXTRACT EQUIPMENT → SPACE RELATIONS
# ============================================================

equipment_rows = []

# We inspect elements spatially contained in each IfcSpace.
for space in spaces:

    space_gid = getattr(
        space,
        "GlobalId",
        ""
    )

    space_name = getattr(
        space,
        "Name",
        ""
    )

    # Inverse relation:
    # IfcRelContainedInSpatialStructure
    try:

        contains = getattr(
            space,
            "ContainsElements",
            []
        ) or []

    except Exception:

        contains = []

    for relation in contains:

        related_elements = getattr(
            relation,
            "RelatedElements",
            []
        ) or []

        for element in related_elements:

            try:

                element_type = (
                    element.is_a()
                )

            except Exception:

                element_type = ""

            equipment_rows.append(
                {
                    "space_global_id":
                        space_gid,

                    "space_name":
                        space_name,

                    "element_global_id":
                        getattr(
                            element,
                            "GlobalId",
                            ""
                        ),

                    "element_type":
                        element_type,

                    "element_name":
                        getattr(
                            element,
                            "Name",
                            ""
                        ),

                    "element_object_type":
                        getattr(
                            element,
                            "ObjectType",
                            ""
                        ),

                    "element_tag":
                        getattr(
                            element,
                            "Tag",
                            ""
                        )
                }
            )

equipment_space_df = pd.DataFrame(
    equipment_rows
)

equipment_space_df.to_csv(
    RECON_DIR /
    "08_ifc_equipment_space_relations.csv",
    index=False
)

print("\n====================================")
print("EQUIPMENT → SPACE RELATIONS")
print("====================================")

print(
    "Contained elements:",
    len(equipment_space_df)
)

if len(equipment_space_df) > 0:

    print(
        equipment_space_df
        .head(20)
        .to_string(
            index=False
        )
    )

# ============================================================
# 8. SEARCH UUIDs THROUGH IFC ENTITY ATTRIBUTES
# ============================================================

# This is broader than the exact textual search performed
# in Notebook 02. It records which entity attributes were
# inspected and whether any UUID appears.

uuid_evidence = []

for uuid in devices:

    found_evidence = []

    for entity in ifc:

        # We deliberately inspect common descriptive fields.
        fields = {
            "GlobalId":
                getattr(
                    entity,
                    "GlobalId",
                    ""
                ),

            "Name":
                getattr(
                    entity,
                    "Name",
                    ""
                ),

            "Description":
                getattr(
                    entity,
                    "Description",
                    ""
                ),

            "ObjectType":
                getattr(
                    entity,
                    "ObjectType",
                    ""
                ),

            "Tag":
                getattr(
                    entity,
                    "Tag",
                    ""
                )
        }

        for field_name, field_value in fields.items():

            if field_value is None:
                continue

            if uuid.lower() in str(
                field_value
            ).lower():

                found_evidence.append(
                    {
                        "operational_uuid":
                            uuid,

                        "ifc_entity_type":
                            entity.is_a(),

                        "ifc_global_id":
                            getattr(
                                entity,
                                "GlobalId",
                                ""
                            ),

                        "field":
                            field_name,

                        "matched_value":
                            str(
                                field_value
                            ),

                        "evidence_type":
                            "direct_identifier_match"
                    }
                )

    if not found_evidence:

        uuid_evidence.append(
            {
                "operational_uuid":
                    uuid,

                "ifc_entity_type":
                    "",

                "ifc_global_id":
                    "",

                "field":
                    "",

                "matched_value":
                    "",

                "evidence_type":
                    "no_direct_identifier_evidence"
            }
        )

    else:

        uuid_evidence.extend(
            found_evidence
        )

uuid_evidence_df = pd.DataFrame(
    uuid_evidence
)

uuid_evidence_df.to_csv(
    RECON_DIR /
    "09_uuid_ifc_evidence.csv",
    index=False
)

print("\n====================================")
print("UUID EVIDENCE SEARCH")
print("====================================")

print(
    uuid_evidence_df.to_string(
        index=False
    )
)

# ============================================================
# 9. CREATE DEVICE × SPACE CANDIDATE MATRIX
# ============================================================

candidate_rows = []

for uuid in devices:

    for _, space in spaces_df.iterrows():

        candidate_rows.append(
            {
                "operational_uuid":
                    uuid,

                "ifc_space_global_id":
                    space[
                        "ifc_global_id"
                    ],

                "ifc_space_name":
                    space[
                        "ifc_name"
                    ],

                "ifc_space_long_name":
                    space[
                        "ifc_long_name"
                    ],

                # Evidence indicators
                "direct_uuid_match":
                    False,

                "metadata_match":
                    False,

                "equipment_evidence":
                    False,

                "spatial_evidence":
                    False,

                # Initial classification
                "reconciliation_class":
                    "C0_UNRESOLVED",

                "mapping_origin":
                    "none",

                "factual_location_claim":
                    False,

                "approved_for_rdf":
                    False
            }
        )

candidate_matrix = pd.DataFrame(
    candidate_rows
)

# ============================================================
# 10. PROPAGATE DIRECT UUID EVIDENCE
# ============================================================

direct_matches = uuid_evidence_df[
    uuid_evidence_df[
        "evidence_type"
    ]
    ==
    "direct_identifier_match"
]

for _, evidence in direct_matches.iterrows():

    uuid = evidence[
        "operational_uuid"
    ]

    gid = evidence[
        "ifc_global_id"
    ]

    mask = (
        (
            candidate_matrix[
                "operational_uuid"
            ]
            ==
            uuid
        )
        &
        (
            candidate_matrix[
                "ifc_space_global_id"
            ]
            ==
            gid
        )
    )

    candidate_matrix.loc[
        mask,
        "direct_uuid_match"
    ] = True

# ============================================================
# 11. SAVE CANDIDATE MATRIX
# ============================================================

candidate_matrix.to_csv(
    RECON_DIR /
    "10_uuid_space_candidate_matrix.csv",
    index=False
)

print("\n====================================")
print("CANDIDATE MATRIX")
print("====================================")

print(
    "Operational devices:",
    len(devices)
)

print(
    "IFC spaces:",
    len(spaces_df)
)

print(
    "Candidate UUID-space pairs:",
    len(candidate_matrix)
)

# ============================================================
# 12. RECONCILIATION EVIDENCE SUMMARY
# ============================================================

summary_rows = []

for uuid in devices:

    evidence_for_device = (
        uuid_evidence_df[
            uuid_evidence_df[
                "operational_uuid"
            ]
            ==
            uuid
        ]
    )

    direct_count = int(
        (
            evidence_for_device[
                "evidence_type"
            ]
            ==
            "direct_identifier_match"
        ).sum()
    )

    if direct_count > 0:

        classification = (
            "C3_NATIVE_VERIFIED_REVIEW_REQUIRED"
        )

    else:

        classification = (
            "C0_UNRESOLVED"
        )

    summary_rows.append(
        {
            "operational_uuid":
                uuid,

            "direct_identifier_evidence":
                direct_count,

            "initial_reconciliation_class":
                classification,

            "factual_location_claim":
                False,

            "approved_for_rdf":
                False,

            "next_action":
                (
                    "Review IFC spatial/equipment/"
                    "metadata evidence before enrichment"
                )
        }
    )

summary_df = pd.DataFrame(
    summary_rows
)

summary_df.to_csv(
    RECON_DIR /
    "11_reconciliation_evidence_summary.csv",
    index=False
)

print("\n====================================")
print("RECONCILIATION EVIDENCE SUMMARY")
print("====================================")

print(
    summary_df.to_string(
        index=False
    )
)

# ============================================================
# 13. FINAL OUTPUT SUMMARY
# ============================================================

print("\n====================================")
print("EVIDENCE SEARCH COMPLETED")
print("====================================")

print("\nFiles created:")

for output_file in sorted(
    RECON_DIR.glob("*.csv")
):

    print(
        " -",
        output_file.name
    )

print(
    "\nIMPORTANT:"
)

print(
    "No experimental UUID → IfcSpace "
    "mapping has been created."
)

print(
    "The next step is to review the "
    "evidence before approving any "
    "semantic enrichment."
)